# 任务 B：指针追踪问题的约束集校准（v4，Qwen3-4B-Instruct-2507 + vLLM）

本 notebook 做 **任务 B 线**：用**指针追踪（pointer tracking）**题目，筛出 10 条能进 `k` 轴、且在 CoT 侧有成本的约束。
**只看约束遵循 `cons_pass`**；`answer_ok` 只作诊断，不作为任务 B 指标。

> **必须在出结果后人工审查。** 自动判分只能提示候选，不能替代逐条看输出。
> 重点看：标签行是否只出现一次、DA 是否偷写推理、CoT 是否真的逐指针对齐、答案大小写是否被约束污染。

## 与上一版（min-chain）的区别

| 项 | 上一版 | 本版 |
|---|---|---|
| 题目 | 小数加减链 | **指针追踪**，直接复用 `synthesis_q_k_dataset.jsonl` 的题面 |
| 答案 | 小数 | 单个大写标签（如 `R`） |
| 查表字段 | `START/OPS/FIRST/LAST/NEG/MAXDELTA` | `START/D/NEXT/LASTROW/FANIN_START/TABLE_SIZE` |
| 约束数量 | 12 + 4 backup | **12 + 4 backup**，格式轨保留 5 条，查表轨 6 条，样板行 1 条 |
| 数据 | `synthesis_math_dataset.jsonl` | `prompt_test_pointer_dataset.jsonl`（本 notebook 生成） |

## 流程

| 步骤 | 内容 | 产物 |
|---|---|---|
| 1 | 安装 vLLM（Colab / Kaggle） | 环境 |
| 2 | 优先读已上传的 50 题 `prompt_test_pointer_dataset.jsonl`；缺失才从 `synthesis_q_k_dataset.jsonl` 生成 | 固定 50 题 |
| 3 | 定义 12 条候选约束 + 4 条 backup | `CONS` |
| 4 | 加载 4B（T4 用 `dtype=half`） | `llm` |
| 5 | 跑 `F0 + 12 条` × 50 题 × DA/CoT = **1300 条** | `calib_v4_cons.jsonl` |
| 6 | 三门槛 + 逻辑独立性 | `calib_v4_cons_verdicts.jsonl` |
| 7 | 不足 10 条自动跑 backup | 补足 |
| 8 | 人工抽查样本 | 最终 10 条 |

`F0` 是额外 100 条基线，用来算「F0 自然违反率 / 主动生成率」，不占 12 条候选名额。
正式校准是 `12 × 50 × 2 = 1200` 条，加 F0 共 1300 条。

## 运行环境

- 本地 WSL 没有 GPU 时**不要执行**第 4 步之后；数据生成和判分自测可以本地跑。
- Kaggle / Colab 的 T4：`dtype=half`，`gpu_memory_utilization=0.90`，`max_model_len=8192`，`max_num_seqs=32`。
- 模型固定 `Qwen/Qwen3-4B-Instruct-2507`。**不要**换成 base 版或 Thinking 版。
- 该模型是非 thinking 模型，**不要**传 `chat_template_kwargs={"enable_thinking": ...}`。

## 1. 安装 vLLM（Colab / Kaggle 专用）

本地已装好 vLLM 时跳过。Colab 安装后若提示 restart，重启会话，从第 2 格继续。

In [1]:
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

if IN_COLAB or IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm", "pandas"], check=True)
    print("vLLM 安装完成。若提示 restart，请重启会话后从下一格开始。")
else:
    print("非 Colab/Kaggle 环境，跳过安装。")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.2.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


vLLM 安装完成。若提示 restart，请重启会话后从下一格开始。


## 2. 配置

- 数据：优先读已上传的 `prompt_test_pointer_dataset.jsonl`（Kaggle input / 仓库 / `/content/`），固定 50 题；缺失才现场生成。
- 取 `d=5..10`，每个 `d` 8–9 题，共 **50 题**。
- 生成专用数据集 `prompt_test_pointer_dataset.jsonl`，带约束判分字段。
- 采样参数沿用项目统一值：`temperature=0.6`，`top_p=0.95`；DA 256 token，CoT 2048 token。

In [2]:
import os, sys, json, re, time, random, math, gc, glob
from collections import Counter

# 必须在 import vllm 之前设置（AGENTS 第 3 节）
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

TIER = "B"
HF_ID = "Qwen/Qwen3-4B-Instruct-2507"
LOCAL_DIR = "models/Qwen3-4B-Instruct-2507"

PROMPT_VERSION = "v4-pointer-cons-calib"
SEED = 20261107
D_LIST = [5, 6, 7, 8, 9, 10]
N_PER_D = {5: 8, 6: 8, 7: 8, 8: 8, 9: 9, 10: 9}
N_PER_D_TOTAL = sum(N_PER_D.values())
MODES = ("DA", "CoT")
MAX_TOKENS = {"DA": 256, "CoT": 2048}
TEMPERATURE, TOP_P = 0.6, 0.95
TARGET_K = 10

DATASET_NAME = "prompt_test_pointer_dataset.jsonl"

# 已上传的 50 题数据集优先；找不到才现场生成。
# Kaggle 路径形如 /kaggle/input/<dataset-slug>/prompt_test_pointer_dataset.jsonl
DATASET_CANDIDATES = [
    "Experiments/data_generation/" + DATASET_NAME,
    "data_generation/" + DATASET_NAME,
    DATASET_NAME,
    "/content/" + DATASET_NAME,
]
DATASET_CANDIDATES += sorted(glob.glob("/kaggle/input/**/" + DATASET_NAME, recursive=True))
DATASET_CANDIDATES += sorted(glob.glob("/content/**/" + DATASET_NAME, recursive=True))
DATASET_IN = next((p for p in DATASET_CANDIDATES if os.path.isfile(p)), None)

MOUNT_DRIVE = True
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/cisc7404_calib"
elif IN_KAGGLE:
    OUT_DIR = "/kaggle/working"
else:
    OUT_DIR = "Experiments" if os.path.isdir("Experiments") else "."
os.makedirs(OUT_DIR, exist_ok=True)

if os.path.isdir("Experiments/data_generation"):
    DATASET_OUT = "Experiments/data_generation/prompt_test_pointer_dataset.jsonl"
elif os.path.isdir("data_generation"):
    DATASET_OUT = "data_generation/prompt_test_pointer_dataset.jsonl"
else:
    DATASET_OUT = os.path.join(OUT_DIR, "prompt_test_pointer_dataset.jsonl")
os.makedirs(os.path.dirname(DATASET_OUT) or ".", exist_ok=True)

OUT_RAW = os.path.join(OUT_DIR, "calib_v4_cons.jsonl")
OUT_SUMMARY = os.path.join(OUT_DIR, "calib_v4_cons_summary.csv")
OUT_VERDICTS = os.path.join(OUT_DIR, "calib_v4_cons_verdicts.jsonl")
OUT_INDEP_F0 = os.path.join(OUT_DIR, "calib_v4_cons_independence_f0.csv")
OUT_INDEP_CROSS = os.path.join(OUT_DIR, "calib_v4_cons_independence_cross.csv")
OUT_SELECTED = os.path.join(OUT_DIR, "calib_v4_cons_selected.json")

print("MODEL:", HF_ID)
print("DATASET_IN:", DATASET_IN)
print("DATASET_OUT:", DATASET_OUT)
print("OUT_DIR:", OUT_DIR)
print("题目:", N_PER_D_TOTAL, "| 候选约束: 12 (+4 backups) | 生成量: 12 x", N_PER_D_TOTAL, "x 2 + F0 =",
      12 * N_PER_D_TOTAL * 2 + N_PER_D_TOTAL * 2)

MODEL: Qwen/Qwen3-4B-Instruct-2507
DATASET_IN: /kaggle/input/datasets/paradoxengine001/promp-test-pointer-dataset/prompt_test_pointer_dataset.jsonl
DATASET_OUT: /kaggle/working/prompt_test_pointer_dataset.jsonl
OUT_DIR: /kaggle/working
题目: 50 | 候选约束: 12 (+4 backups) | 生成量: 12 x 50 x 2 + F0 = 1300


## 3. 生成指针追踪数据集

- **优先用已上传的 `prompt_test_pointer_dataset.jsonl`**（Kaggle input、仓库 `Experiments/data_generation/`、或 `/content/`）。
- 找不到才现场生成 50 题，并写到 `DATASET_OUT`。
- 已上传文件不会被覆盖，保证结果可复现。
- 每个 `(d, k, i)` 的题面唯一，`qid` 用 `d/k/i` 保证不冲突。
- 生成字段：
  - `start`：起点标签
  - `next`：走 1 步后的标签
  - `second`：走 2 步后的标签
  - `lastrow`：表格最后一行（`Z -> ?`）的右值
  - `firstrow`：表格第一行（`A -> ?`）的右值
  - `fanin_start` / `fanin_next`：有多少行的右值等于 `start` / `next`
  - `table_size`：表格行数（本数据固定 24）
  - `odd_d`：步数是否为奇数
  - `path`：完整路径，供人工审查

这些字段都只要求读题/扫表，不需要算最终答案，和 `answer_ok` 解耦。

In [3]:
LABELS = list("ABCDEFGHJKLMNPQRSTUVWXYZ")  # 24 个，去掉 I / O


# ---------- 内置生成器：源文件缺失时使用，逻辑参考 synthesis_dataset.ipynb ----------
def _make_table(rng, n=len(LABELS), min_cycle_len=2):
    labels = LABELS[:n]
    cycle_len = rng.randint(min_cycle_len, n)
    cycle = rng.sample(labels, cycle_len)
    table = {}
    for i, node in enumerate(cycle):
        table[node] = cycle[(i + 1) % cycle_len]
    rest = [node for node in labels if node not in table]
    rng.shuffle(rest)
    attached = list(cycle)
    for node in rest:
        table[node] = rng.choice(attached)
        attached.append(node)
    return {node: table[node] for node in labels}


def _follow(table, start, steps):
    node = start
    for _ in range(steps):
        node = table[node]
    return node


def _table_text(table):
    return "\n".join("%s -> %s" % (node, table[node]) for node in LABELS)


def make_serial_question(rng, d):
    table = _make_table(rng, len(LABELS), d + 1)
    start = rng.choice(LABELS)
    ans = _follow(table, start, d)
    step_word = "step" if d == 1 else "steps"
    q = ("The table below defines a single pointer for each entry: from an entry, the pointer tells you which entry comes next.\n\n"
         "%s\n\n"
         "Start at %s. Follow the pointers for exactly %s %s.\n"
         "Question: which entry do you end at?") % (_table_text(table), start, d, step_word)
    return dict(q=q, ans=ans, d=d, W=d, k=0, i=0, table=table)


# ---------- 解析成约束判分字段 ----------
def parse_pointer_record(rec):
    q = rec["q"]
    m = re.search(r"Start at ([A-Z])\.", q)
    if not m:
        raise ValueError("找不到起点: " + q[:200])
    start = m.group(1)
    table = dict(rec["table"])
    if set(table) != set(LABELS) or len(table) != len(LABELS):
        raise ValueError("表不是 24 个标签")
    if start not in table:
        raise ValueError("起点不在表里")

    path = [start]
    node = start
    for _ in range(int(rec["d"])):
        node = table[node]
        path.append(node)
    ans = str(rec["ans"]).strip()
    if node != ans:
        raise ValueError("答案与表不一致: %s vs %s" % (node, ans))

    nxt = table[start]
    second = table[nxt]
    firstrow = table[LABELS[0]]
    lastrow = table[LABELS[-1]]
    return {
        "qid": "d%s_k%s_i%s" % (rec["d"], rec.get("k", 0), rec.get("i", 0)),
        "d": int(rec["d"]),
        "k": int(rec.get("k", 0)),
        "i": int(rec.get("i", 0)),
        "W": int(rec.get("W", rec["d"])),
        "q": q,
        "ans": ans,
        "table": table,
        "start": start,
        "next": nxt,
        "second": second,
        "lastrow": lastrow,
        "firstrow": firstrow,
        "fanin_start": sum(1 for v in table.values() if v == start),
        "fanin_next": sum(1 for v in table.values() if v == nxt),
        "table_size": len(table),
        "odd_d": bool(int(rec["d"]) % 2),
        "path": path,
    }


# ---------- 读取已上传数据集并重算派生字段；找不到才现场生成 ----------
def normalize_pointer_record(rec):
    """兼容旧 JSONL：只信 q/table/d/ans，其余派生字段全部重算。"""
    q = rec["q"]
    m = re.search(r"Start at ([A-Z])\.", q)
    if not m:
        raise ValueError("找不到起点: " + q[:200])
    start = m.group(1)
    table = {str(k): str(v) for k, v in rec["table"].items()}
    if set(table) != set(LABELS) or len(table) != len(LABELS):
        raise ValueError("表不是 24 个标签: " + str(rec.get("qid")))
    if start not in table:
        raise ValueError("起点不在表里: " + start)

    d = int(rec["d"])
    path = [start]
    node = start
    for _ in range(d):
        node = table[node]
        path.append(node)
    ans = str(rec["ans"]).strip()
    if node != ans:
        raise ValueError("答案与表不一致: %s vs %s" % (node, ans))

    nxt = table[start]
    return {
        "qid": str(rec.get("qid", "d%s_k%s_i%s" % (d, rec.get("k", 0), rec.get("i", 0)))),
        "d": d,
        "k": int(rec.get("k", 0)),
        "i": int(rec.get("i", 0)),
        "W": int(rec.get("W", d)),
        "q": q,
        "ans": ans,
        "table": table,
        "start": start,
        "next": nxt,
        "second": table[nxt],
        "lastrow": table[LABELS[-1]],
        "firstrow": table[LABELS[0]],
        "fanin_start": sum(1 for v in table.values() if v == start),
        "fanin_next": sum(1 for v in table.values() if v == nxt),
        "table_size": len(table),
        "odd_d": bool(d % 2),
        "path": path,
    }


if DATASET_IN:
    src_rows = [json.loads(l) for l in open(DATASET_IN, encoding="utf-8") if l.strip()]
    QUESTIONS = [normalize_pointer_record(r) for r in src_rows]
    print("使用已上传数据集:", DATASET_IN, "| 行数:", len(QUESTIONS))
    print("不覆盖该文件。要重新生成，把 DATASET_IN 置空。")
else:
    rng = random.Random(SEED)
    QUESTIONS = []
    for d in D_LIST:
        for j in range(N_PER_D[d]):
            rec = make_serial_question(rng, d)
            rec["i"] = j + 1
            QUESTIONS.append(parse_pointer_record(rec))
    with open(DATASET_OUT, "w", encoding="utf-8") as f:
        for q in QUESTIONS:
            f.write(json.dumps(q, ensure_ascii=False) + "\n")
    print("源数据集缺失，已现场生成:", DATASET_OUT, "| 行数:", len(QUESTIONS))

assert len(QUESTIONS) == N_PER_D_TOTAL, "题目数不是 %s: %s" % (N_PER_D_TOTAL, len(QUESTIONS))
assert len({q["qid"] for q in QUESTIONS}) == len(QUESTIONS)
assert len({q["q"] for q in QUESTIONS}) == len(QUESTIONS)

print("按 d:", dict(sorted(Counter(q["d"] for q in QUESTIONS).items())))
print("答案标签分布（前 8）:", Counter(q["ans"] for q in QUESTIONS).most_common(8))
print("\n第一题:", QUESTIONS[0]["qid"], "d=%s ans=%s" % (QUESTIONS[0]["d"], QUESTIONS[0]["ans"]))
print(QUESTIONS[0]["q"])
print({k: QUESTIONS[0][k] for k in ("start", "next", "second", "lastrow", "firstrow",
                                     "fanin_start", "fanin_next", "table_size", "odd_d")})

使用已上传数据集: /kaggle/input/datasets/paradoxengine001/promp-test-pointer-dataset/prompt_test_pointer_dataset.jsonl | 行数: 50
不覆盖该文件。要重新生成，把 DATASET_IN 置空。
按 d: {5: 8, 6: 8, 7: 8, 8: 8, 9: 9, 10: 9}
答案标签分布（前 8）: [('D', 5), ('M', 4), ('F', 4), ('V', 4), ('H', 4), ('J', 3), ('Z', 2), ('C', 2)]

第一题: d5_k1_i9 d=5 ans=R
The table below defines a single pointer for each entry: from an entry, the pointer tells you which entry comes next.

A -> Z
B -> X
C -> W
D -> E
E -> U
F -> G
G -> A
H -> N
J -> N
K -> G
L -> W
M -> E
N -> C
P -> A
Q -> D
R -> Q
S -> K
T -> Q
U -> H
V -> W
W -> F
X -> F
Y -> S
Z -> R

Start at S. Follow the pointers for exactly 5 steps.
Question: which entry do you end at?
{'start': 'S', 'next': 'K', 'second': 'G', 'lastrow': 'R', 'firstrow': 'Z', 'fanin_start': 1, 'fanin_next': 1, 'table_size': 24, 'odd_d': True}


## 4. 12 条候选约束 + 4 条 backup

### 4.1 从 min-chain 映射过来

指针题答案是**标签**，不是小数，所以两条查表约束要按任务重定义（`record.md` 第 4 节：查表类必须按新任务字段重定义）。
映射关系：

| 上一版 | 本版 | 说明 |
|---|---|---|
| `DEC2` | `LOWER` | 数字两位小数不适用；改成「最终标签必须小写」 |
| `UPPER` | `UPPER` | 全大写；与 `LOWER` 互斥，选择时只能留一条 |
| `PAREN` | `PAREN` | 最终标签加括号 |
| `NO_COMMA` | `NO_COMMA` | 不出现逗号 |
| `NO_EQUALS` | `NO_EQUALS` | 除标签行/答案行外不出现等号 |
| `START` | `START` | 起点标签 |
| `OPS` | `D` | 步数 |
| `FIRST` | `NEXT` | 走 1 步后的标签 |
| `LAST` | `LASTROW` | 表格最后一行的右值 |
| `NEG` | `FANIN_START` | 有多少行指向起点 |
| `MAXDELTA` | `TABLE_SIZE` | 表格行数（24） |
| `STATUS` | `STATUS` | `STATUS=DONE` 样板行 |

### 4.2 12 条候选

| ID | 轨 | 类型 | 规则摘要 | 预期 |
|---|---|---|---|---|
| `LOWER` | format | require | 最终标签小写 | 可能通过 |
| `UPPER` | format | require | 全大写 | 与 `LOWER` 互斥 |
| `PAREN` | format | require | 最终标签加括号 | 可能通过 |
| `NO_COMMA` | format | ban | 不出现逗号 | DA 免费，预计淘汰 |
| `NO_EQUALS` | format | ban | 除标签行外不出现等号 | DA 免费，预计淘汰 |
| `START` | lookup | require | `START=起点` | 主力 |
| `D` | lookup | require | `D=步数` | 主力 |
| `NEXT` | lookup | require | `NEXT=1 步后标签` | 主力 |
| `LASTROW` | lookup | require | `LASTROW=最后一行右值` | 主力 |
| `FANIN_START` | lookup | require | `FANIN_START=入度` | 主力 |
| `TABLE_SIZE` | lookup | require | `TABLE_SIZE=24` | 可能偏易 |
| `STATUS` | boilerplate | require | `STATUS=DONE` | 预期有 CoT 成本 |

backup：`SECOND / ODD_D / FIRSTROW / FANIN_NEXT`，不足 10 条时自动补跑。

### 4.3 门槛

| 门槛 | 判据 |
|---|---|
| 结构 | `base_ok >= 0.90`，`placement_ok >= 0.90`，`trunc < 5%`，`copy_flag <= 1%`，DA `da_other <= 20%` |
| 非空转 | 禁令类：F0(DA) 自然违反率 `>= 0.15`；要求类：F0(DA) 主动生成率 `<= 0.90` |
| DA 可遵循 | 单条 `cons_pass(DA) >= 0.95` |
| CoT 有成本 | `pass_DA - pass_CoT >= 0.15`，且 gap 对 d 的斜率 `<= 0.02` |
| 独立性 | `P(B 不通过 | A 通过) >= 0.05`；否则 B 视为冗余 |

### 4.4 判分

- `base_ok`：全文恰好一个 `####ANSWER=` 行，后面是**单个标签**（允许括号包裹）。位置不参与判断。
- `placement_ok`：`####ANSWER=` 必须是全文最后一行。keyword 行写在它后面即判失败。
- `cons_only`：只看约束值本身；`cons_ok = base_ok and cons_only`；`strict_cons_ok = cons_ok and placement_ok`。
- 门槛和 gap 用 `strict_cons_ok`；`cons_only` / `base_ok` / `placement_ok` 用来归因。
- `per_cons_all` 在每一行上检查全部候选，供 F0 自然违反率和独立性矩阵使用。
- DA 多余行拆两类：`da_trace`（`X -> Y` / `Step n` / `Entry reached` 等步骤行）和 `da_other`（其他）。结构门槛只看 `da_other`。
- `copy_flag`：出现 3 个以上 `####ANSWER=`，或出现 `<label>` 之类占位符。

In [4]:
MARK = "####ANSWER="
LABEL_KEYS = ("START", "D", "NEXT", "LASTROW", "FANIN_START", "TABLE_SIZE", "STATUS",
              "SECOND", "ODD_D", "FIRSTROW", "FANIN_NEXT")
KEYLINE_RE = re.compile(r"^(" + "|".join(re.escape(k) for k in LABEL_KEYS) + r")\s*=")


def lines_of(t):
    return [l.strip() for l in str(t).strip().splitlines() if l.strip()]


def marker_lines(t):
    return [l for l in lines_of(t) if l.startswith(MARK)]


def final_raw(t):
    # 取最后一个 ####ANSWER= 行；不再要求它是全文最后一行（位置单独判 placement_ok）
    ls = marker_lines(t)
    if not ls:
        return None
    return ls[-1][len(MARK):].strip()


def marker_last_index(t):
    idx = [i for i, l in enumerate(lines_of(t)) if l.startswith(MARK)]
    return idx[-1] if idx else None


def is_trace_line(line):
    if re.fullmatch(r"[A-Za-z]\s*->\s*[A-Za-z]", line):
        return True
    if re.match(r"^Step\s*\d+\s*[:.]", line, re.I):
        return True
    if re.match(r"^(Entry reached|Final entry|Start at)\b", line, re.I):
        return True
    return False


def unwrap(s):
    if s is None:
        return None
    s = str(s).strip()
    for _ in range(2):
        if len(s) >= 2 and ((s[0] == "(" and s[-1] == ")") or (s[0] == "[" and s[-1] == "]")):
            s = s[1:-1].strip()
    return s


def final_label(t):
    s = unwrap(final_raw(t))
    if s is not None and re.fullmatch(r"[A-Za-z]", s):
        return s
    return None


def kv_values(t, key):
    pat = re.compile(r"^" + re.escape(key) + r"\s*=\s*(.*?)\s*$")
    out = []
    for line in lines_of(t):
        m = pat.fullmatch(line)
        if m:
            out.append(m.group(1).strip())
    return out


def one_kv_label(t, key, expected):
    vals = kv_values(t, key)
    if len(vals) != 1:
        return False
    v = unwrap(vals[0])
    return v is not None and re.fullmatch(r"[A-Za-z]", v) is not None and v.upper() == str(expected).upper()


def one_kv_number(t, key, expected, tol=1e-6):
    vals = kv_values(t, key)
    if len(vals) != 1:
        return False
    try:
        x = float(unwrap(vals[0]))
    except Exception:
        return False
    return abs(x - expected) <= tol


def chk_LOWER(t, r):
    v = final_label(t)
    return v is not None and v.islower()


def chk_UPPER(t, r):
    return re.search(r"[a-z]", t) is None


def chk_PAREN(t, r):
    raw = final_raw(t)
    if raw is None:
        return False
    raw = raw.strip()
    return len(raw) >= 3 and raw[0] == "(" and raw[-1] == ")" and final_label(t) is not None


def chk_NO_COMMA(t, r):
    return "," not in t


def chk_NO_EQUALS(t, r):
    for line in lines_of(t):
        if "=" not in line:
            continue
        if line.startswith(MARK):
            continue
        if KEYLINE_RE.match(line):
            continue
        return False
    return True


def chk_START(t, r):
    return one_kv_label(t, "START", r["start"])


def chk_D(t, r):
    return one_kv_number(t, "D", r["d"])


def chk_NEXT(t, r):
    return one_kv_label(t, "NEXT", r["next"])


def chk_LASTROW(t, r):
    return one_kv_label(t, "LASTROW", r["lastrow"])


def chk_FANIN_START(t, r):
    return one_kv_number(t, "FANIN_START", r["fanin_start"])


def chk_TABLE_SIZE(t, r):
    return one_kv_number(t, "TABLE_SIZE", r["table_size"])


def chk_STATUS(t, r):
    return len([l for l in lines_of(t) if re.fullmatch(r"STATUS\s*=\s*DONE", l)]) == 1


def chk_SECOND(t, r):
    return one_kv_label(t, "SECOND", r["second"])


def chk_ODD_D(t, r):
    vals = [unwrap(v).upper() for v in kv_values(t, "ODD_D")]
    expected = "YES" if r["odd_d"] else "NO"
    return len(vals) == 1 and vals[0] == expected


def chk_FIRSTROW(t, r):
    return one_kv_label(t, "FIRSTROW", r["firstrow"])


def chk_FANIN_NEXT(t, r):
    return one_kv_number(t, "FANIN_NEXT", r["fanin_next"])


BEFORE = " This line must appear before the ####ANSWER= line."

CONS = {
    "F0": dict(track="base", kind="base", keys=(), text=None, check=lambda t, r: True),

    "LOWER": dict(track="format", kind="require", keys=(), check=chk_LOWER,
                  text="The final answer label after ####ANSWER= must be written in lowercase."),
    "UPPER": dict(track="format", kind="require", keys=(), check=chk_UPPER,
                  text="Every letter in the response must be uppercase."),
    "PAREN": dict(track="format", kind="require", keys=(), check=chk_PAREN,
                  text="The final answer label after ####ANSWER= must be enclosed in parentheses."),
    "NO_COMMA": dict(track="format", kind="ban", keys=(), check=chk_NO_COMMA,
                     text="Never use a comma character anywhere in the response."),
    "NO_EQUALS": dict(track="format", kind="ban", keys=(), check=chk_NO_EQUALS,
                      text="Never use the equals sign anywhere in the response, except in the line that begins with ####ANSWER= and in the keyword lines that these rules require."),

    "START": dict(track="lookup", kind="require", keys=("START",), check=chk_START,
                  text="Include exactly one line that begins with START= followed by the starting entry label." + BEFORE),
    "D": dict(track="lookup", kind="require", keys=("D",), check=chk_D,
              text="Include exactly one line that begins with D= followed by the number of pointer steps stated in the question." + BEFORE),
    "NEXT": dict(track="lookup", kind="require", keys=("NEXT",), check=chk_NEXT,
                 text="Include exactly one line that begins with NEXT= followed by the entry reached after exactly one step from the starting entry. Always count from the starting entry, never from a later entry." + BEFORE),
    "LASTROW": dict(track="lookup", kind="require", keys=("LASTROW",), check=chk_LASTROW,
                    text="Include exactly one line that begins with LASTROW= followed by the label on the right side of the row whose left side is the last label of the table. This is not the final answer." + BEFORE),
    "FANIN_START": dict(track="lookup", kind="require", keys=("FANIN_START",), check=chk_FANIN_START,
                        text="Include exactly one line that begins with FANIN_START= followed by how many rows of the pointer table have the starting entry on their right side." + BEFORE),
    "TABLE_SIZE": dict(track="lookup", kind="require", keys=("TABLE_SIZE",), check=chk_TABLE_SIZE,
                       text="Include exactly one line that begins with TABLE_SIZE= followed by the number of entry lines in the pointer table. Count only lines of the form LABEL -> LABEL." + BEFORE),

    "STATUS": dict(track="boilerplate", kind="require", keys=("STATUS",), check=chk_STATUS,
                   text="Include exactly one line that consists only of the word STATUS, an equals sign, and the word DONE." + BEFORE),

    "SECOND": dict(track="lookup", kind="require", keys=("SECOND",), check=chk_SECOND,
                   text="Include exactly one line that begins with SECOND= followed by the entry reached after exactly two steps from the starting entry." + BEFORE),
    "ODD_D": dict(track="lookup", kind="require", keys=("ODD_D",), check=chk_ODD_D,
                  text="Include exactly one line that begins with ODD_D= followed by YES if the number of pointer steps is odd, or NO if it is even." + BEFORE),
    "FIRSTROW": dict(track="lookup", kind="require", keys=("FIRSTROW",), check=chk_FIRSTROW,
                     text="Include exactly one line that begins with FIRSTROW= followed by the label on the right side of the first row of the pointer table." + BEFORE),
    "FANIN_NEXT": dict(track="lookup", kind="require", keys=("FANIN_NEXT",), check=chk_FANIN_NEXT,
                       text="Include exactly one line that begins with FANIN_NEXT= followed by how many rows of the pointer table have the one-step entry on their right side." + BEFORE),
}

CANDIDATES = ["LOWER", "UPPER", "PAREN", "NO_COMMA", "NO_EQUALS",
              "START", "D", "NEXT", "LASTROW", "FANIN_START", "TABLE_SIZE", "STATUS"]
BACKUP_CANDIDATES = ["SECOND", "ODD_D", "FIRSTROW", "FANIN_NEXT"]
ALL_CANDIDATES = CANDIDATES + BACKUP_CANDIDATES
INCOMPATIBLE = {frozenset(("LOWER", "UPPER"))}


# ---- 判分函数自测（不需要 GPU）----
_r = dict(d=6, ans="R", start="S", next="K", second="R", lastrow="X", firstrow="Z",
          fanin_start=1, fanin_next=2, table_size=24, odd_d=False)
_t_all = ("START=S\nD=6\nNEXT=K\nLASTROW=X\nFANIN_START=1\nTABLE_SIZE=24\n"
          "STATUS=DONE\n####ANSWER=(r)")
assert chk_LOWER(_t_all, _r) and not chk_UPPER(_t_all, _r)
assert chk_PAREN(_t_all, _r) and chk_NO_COMMA(_t_all, _r) and chk_NO_EQUALS(_t_all, _r)
assert chk_START(_t_all, _r) and chk_D(_t_all, _r) and chk_NEXT(_t_all, _r)
assert chk_LASTROW(_t_all, _r) and chk_FANIN_START(_t_all, _r) and chk_TABLE_SIZE(_t_all, _r)
assert chk_STATUS(_t_all, _r)
assert not chk_SECOND(_t_all, _r) and not chk_ODD_D(_t_all, _r)
assert not chk_FIRSTROW(_t_all, _r) and not chk_FANIN_NEXT(_t_all, _r)

assert chk_UPPER("####ANSWER=R", _r) and not chk_LOWER("####ANSWER=R", _r)
assert not chk_PAREN("####ANSWER=R", _r)
assert not chk_NO_COMMA("A, B\n####ANSWER=R", _r)
assert not chk_NO_EQUALS("A -> B = C\n####ANSWER=R", _r)
assert not chk_D("D=7\n####ANSWER=R", _r)
assert not chk_NEXT("NEXT=Q\n####ANSWER=R", _r)
assert not chk_LASTROW("LASTROW=Z\n####ANSWER=R", _r)
assert not chk_FANIN_START("FANIN_START=2\n####ANSWER=R", _r)
assert not chk_STATUS("STATUS=DONE\nSTATUS=DONE\n####ANSWER=R", _r)

# 位置解耦自测
assert final_label("D=6\n####ANSWER=R") == "R"      # marker 不在最后一行也能解析
assert final_label("####ANSWER=R\nD=6") == "R"
assert marker_last_index("D=6\n####ANSWER=R") == 1
assert marker_last_index("####ANSWER=R\nD=6") == 0
assert is_trace_line("F -> L") and is_trace_line("Step 3: F -> L")
assert not is_trace_line("D=6")

print("判分函数自测通过；候选:", CANDIDATES)
print("backup:", BACKUP_CANDIDATES)
print("互斥对:", [sorted(x) for x in INCOMPATIBLE])

判分函数自测通过；候选: ['LOWER', 'UPPER', 'PAREN', 'NO_COMMA', 'NO_EQUALS', 'START', 'D', 'NEXT', 'LASTROW', 'FANIN_START', 'TABLE_SIZE', 'STATUS']
backup: ['SECOND', 'ODD_D', 'FIRSTROW', 'FANIN_NEXT']
互斥对: [['LOWER', 'UPPER']]


## 5. Prompt 拼装

- `DA` preamble：只允许规则要求的行，**明确禁止 `X -> Y` 行、禁止抄表格、禁止写步骤**。
- `CoT` preamble：先逐步写出每次指针跳转后的标签，再给最终答案。
- `cons` 对 DA / CoT **完全相同**；两种模式只靠 system prompt 的 preamble 区分。
- **顺序固定**：所有 keyword 行在前，`####ANSWER=` 是全文最后一行。规则顺序 + 模板 + 例子一起约束。
- 标签行用「must appear before the ####ANSWER= line」正向表述。

In [5]:
CoT_PREAMBLE = ("Solve the problem step by step. Before the final answer, explicitly write the "
                "entry reached after each pointer step, one per line. Then apply the rules below to the "
                "final answer.\n")
DA_PREAMBLE = ("Solve the problem silently. Your response must contain nothing except the lines required by "
               "the rules below: no calculations, no intermediate values, no explanation and no comments about "
               "the problem. Do not write any line of the form X -> Y. Do not copy rows from the pointer table. "
               "Do not show any pointer step. Any additional text counts as an error.\n")
PREAMBLE = {"DA": DA_PREAMBLE, "CoT": CoT_PREAMBLE}

BASE_RULE = ("Write every keyword line required by the rules below first, in the order the rules are given. "
             "The very last line of your response must begin with ####ANSWER= immediately followed by "
             "the final entry label as a single uppercase letter (unless another rule says otherwise). "
             "Nothing may follow that line.")

OUTPUT_TEMPLATE = ("Required output shape (the values below are only examples):\n"
                   "START=R\n"
                   "D=5\n"
                   "... (other keyword lines required by the rules) ...\n"
                   "####ANSWER=R\n"
                   "With no keyword rules, the whole response is one line: ####ANSWER=R")

# DA 单行样例：防止模型抄表格
DA_SINGLE_LINE = ("Example of a complete DA response when no keyword rules apply:\n"
                  "####ANSWER=R")


def build_cons(ids, rec):
    rules = [BASE_RULE]
    for cid in ids:
        if cid == "F0":
            continue
        rules.append(CONS[cid]["text"])
    parts = ["Rules:\n" + "\n".join("R%s. %s" % (j + 1, r) for j, r in enumerate(rules))]
    if any(c != "F0" for c in ids):
        parts.append(OUTPUT_TEMPLATE)
    else:
        parts.append(DA_SINGLE_LINE)
    return "\n\n".join(parts)


def build_messages(mode, ids, q, rec):
    return [
        {"role": "system", "content": PREAMBLE[mode] + build_cons(ids, rec)},
        {"role": "user", "content": q["q"]},
    ]


def make_jobs(stage, questions, modes, cons_lists):
    jobs = []
    for mode in modes:
        for ids in cons_lists:
            cid = "+".join(ids)
            for q in questions:
                rec = {
                    "id": "%s|%s|%s|%s|%s|%s" % (PROMPT_VERSION, TIER, stage, mode, cid, q["qid"]),
                    "stage": stage,
                    "mode": mode,
                    "cons_id": cid,
                    "cons_ids": list(ids),
                    "k": len([c for c in ids if c != "F0"]),
                    "qid": q["qid"],
                    "d": q["d"],
                    "W": q["W"],
                    "q": q["q"],
                    "ans": q["ans"],
                    "start": q["start"],
                    "next": q["next"],
                    "second": q["second"],
                    "lastrow": q["lastrow"],
                    "firstrow": q["firstrow"],
                    "fanin_start": q["fanin_start"],
                    "fanin_next": q["fanin_next"],
                    "table_size": q["table_size"],
                    "odd_d": q["odd_d"],
                    "prompt_version": PROMPT_VERSION,
                    "tier": TIER,
                    "max_tokens": MAX_TOKENS[mode],
                }
                rec["messages"] = build_messages(mode, ids, q, rec)
                jobs.append(rec)
    return jobs


# ---- prompt 自测 ----
_jobs = make_jobs("selftest", QUESTIONS[:1], MODES, [["F0"], ["START"], ["LOWER", "PAREN"]])
assert len(_jobs) == 2 * 3
for j in _jobs:
    sys_text = j["messages"][0]["content"]
    assert "<label>" not in sys_text and "<answer>" not in sys_text
print("prompt 自测通过；示例 system prompt（START, DA）：")
print(_jobs[1]["messages"][0]["content"])

prompt 自测通过；示例 system prompt（START, DA）：
Solve the problem silently. Your response must contain nothing except the lines required by the rules below: no calculations, no intermediate values, no explanation and no comments about the problem. Do not write any line of the form X -> Y. Do not copy rows from the pointer table. Do not show any pointer step. Any additional text counts as an error.
Rules:
R1. Write every keyword line required by the rules below first, in the order the rules are given. The very last line of your response must begin with ####ANSWER= immediately followed by the final entry label as a single uppercase letter (unless another rule says otherwise). Nothing may follow that line.
R2. Include exactly one line that begins with START= followed by the starting entry label. This line must appear before the ####ANSWER= line.

Required output shape (the values below are only examples):
START=R
D=5
... (other keyword lines required by the rules) ...
####ANSWER=R
With no keywor

## 6. 加载 vLLM（Qwen3-4B-Instruct-2507）

T4 / Turing 不支持 bf16，必须 `dtype="half"`；A100/L4 等可以走默认 bf16。
`Qwen3-4B-Instruct-2507` 是非 thinking 模型，`CHAT_KW` 保持空字典。

In [6]:
import torch
from vllm import LLM, SamplingParams

if not torch.cuda.is_available():
    raise RuntimeError("没有 CUDA。请到 Kaggle / Colab 的 GPU 运行时执行这一格。")

model_path = LOCAL_DIR if os.path.isdir(LOCAL_DIR) else HF_ID
cap = torch.cuda.get_device_capability(0)
mem_gib = torch.cuda.get_device_properties(0).total_memory / 2**30

kw = dict(
    model=model_path,
    max_model_len=8192,
    gpu_memory_utilization=0.90,
    max_num_seqs=32,
)
if cap[0] < 8:
    kw["dtype"] = "half"

print("LLM kwargs:", kw)
print("GPU: %.1f GiB, capability %s" % (mem_gib, cap))
llm = LLM(**kw)

# Qwen3-4B-Instruct-2507 是非 thinking 模型：不要传 enable_thinking
CHAT_KW = {}
SP = {m: SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS[m]) for m in MAX_TOKENS}
print("SamplingParams:", {m: (TEMPERATURE, TOP_P, MAX_TOKENS[m]) for m in MAX_TOKENS})

W1107 15:15:25.370000 49 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1107 15:15:25.371000 49 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1107 15:15:25.373000 49 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


LLM kwargs: {'model': 'Qwen/Qwen3-4B-Instruct-2507', 'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 32, 'dtype': 'half'}
GPU: 14.6 GiB, capability (7, 5)
(MainProcess pid=49) INFO 11-07 15:15:45 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 32, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Instruct-2507'}


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

(MainProcess pid=49) INFO 11-07 15:16:05 [model.py:744] Resolved architecture: Qwen3ForCausalLM
(MainProcess pid=49) WARNING 11-07 15:16:05 [model.py:2516] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=49) INFO 11-07 15:16:05 [model.py:2170] Using max model len 8192
(MainProcess pid=49) INFO 11-07 15:16:05 [scheduler.py:310] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/3 [00:00<?, ?it/s]

... (17 outputs left out)

Capturing CUDA graphs (FULL): 100%|██████████| 7/7 [00:00<00:00, 15.05it/s]


(EngineCore pid=205) INFO 11-07 15:18:34 [model_runner.py:1163] Graph capturing finished in 3 secs, took 0.07 GiB
(EngineCore pid=205) INFO 11-07 15:18:34 [gpu_worker.py:1003] CUDA graph pool memory: 0.07 GiB (actual), 0.13 GiB (estimated), difference: 0.06 GiB (91.2%).
(EngineCore pid=205) INFO 11-07 15:18:34 [gpu_worker.py:1066] Free memory on device (14.41/14.56 GiB) on startup. Desired GPU memory utilization is (0.9, 13.11 GiB). Actual usage is 7.7 GiB for consumed memory (weights + non-torch), 0.91 GiB for peak activation, and 0.07 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=4595939533` (4.28 GiB) to fit into requested memory, or `--kv-cache-memory=5999242240` (5.59 GiB) to fully utilize gpu memory. Current kv cache memory in use is 4.49 GiB.
(EngineCore pid=205) INFO 11-07 15:18:35 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=205) INFO 11-07 15:18:36 [c

Parse safetensors files: 100%|██████████| 3/3 [00:00<00:00, 11.12it/s]


(EngineCore pid=205) INFO 11-07 15:18:37 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(MainProcess pid=49) INFO 11-07 15:18:38 [hf.py:736] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
SamplingParams: {'DA': (0.6, 0.95, 256), 'CoT': (0.6, 0.95, 2048)}


## 7. 生成器

- 结果追加写 JSONL，按 `id` 断点续跑；中断后重跑会自动跳过已完成的行。
- 记录 `prompt_tokens / n_tokens / finish_reason / trunc / batch_sec`。
- 同时把 `per_cons / per_cons_all / base_ok / copy_flag` 判分字段落盘。
- DA 每批 64，CoT 每批 32。

In [7]:
def load_done(path):
    if not os.path.exists(path):
        return set()
    with open(path, encoding="utf-8") as f:
        return {json.loads(l)["id"] for l in f if l.strip()}


def run_jobs(jobs, path=OUT_RAW):
    done = load_done(path)
    todo = [j for j in jobs if j["id"] not in done]
    print("总计 %s，已完成 %s，待跑 %s" % (len(jobs), len(jobs) - len(todo), len(todo)))
    for mode in MODES:
        part = [j for j in todo if j["mode"] == mode]
        batch = 64 if mode == "DA" else 32
        for s in range(0, len(part), batch):
            chunk = part[s:s + batch]
            t0 = time.time()
            outs = llm.chat([j["messages"] for j in chunk], SP[mode], use_tqdm=False, **CHAT_KW)
            dt = time.time() - t0
            with open(path, "a", encoding="utf-8") as f:
                for j, o in zip(chunk, outs):
                    c = o.outputs[0]
                    rec = dict(j)
                    rec.update(
                        text=c.text,
                        prompt_tokens=len(o.prompt_token_ids),
                        n_tokens=len(c.token_ids),
                        finish_reason=c.finish_reason,
                        trunc=(c.finish_reason == "length"),
                        batch_sec=round(dt, 2),
                        batch_size=len(chunk),
                    )
                    sc = score(rec)
                    rec.update({k: sc[k] for k in (
                        "base_ok", "answer_ok", "placement_ok", "cons_only", "cons_ok", "strict_cons_ok",
                        "per_cons", "per_cons_all", "da_extra", "da_trace", "da_other",
                        "cot_trace_ok", "copy_flag", "n_lines", "n_commas")})
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            print("[%s] %s/%s  batch %.1fs" % (mode, s + len(chunk), len(part), dt))


def score(rec):
    t = rec["text"]
    mode = rec["mode"]
    is_da = mode == "DA"
    lines = lines_of(t)
    label = final_label(t)
    mlines = marker_lines(t)

    # base_ok 只看答案行本身是否合法；位置单独用 placement_ok 判
    base_ok = len(mlines) == 1 and label is not None
    placement_ok = marker_last_index(t) == len(lines) - 1
    answer_ok = label is not None and label.upper() == str(rec["ans"]).upper()

    per_all = {c: bool(CONS[c]["check"](t, rec)) for c in ALL_CANDIDATES}
    ids = [c for c in rec.get("cons_ids", ["F0"]) if c != "F0"]
    per = {c: per_all[c] for c in ids}
    cons_only = all(per.values()) if per else True

    allowed = set()
    for c in ids:
        allowed.update(CONS[c].get("keys", ()))

    other = []
    for line in lines:
        if line.startswith(MARK):
            continue
        km = KEYLINE_RE.match(line)
        if km and km.group(1) in allowed:
            continue
        other.append(line)

    # DA 拆分：抄表/步骤行 vs 其他多余文本
    da_trace = bool(is_da and any(is_trace_line(l) for l in other))
    da_other = bool(is_da and any(not is_trace_line(l) for l in other))
    da_extra = bool(is_da and other)

    trace = [l for l in lines if not l.startswith(MARK) and not KEYLINE_RE.match(l)]
    cot_trace_ok = (not is_da) and len(trace) >= max(1, rec["d"] // 2)
    copy_flag = len(mlines) >= 3 or re.search(r"<(?:answer|label|number|value)>", t, re.I) is not None

    return {
        "base_ok": bool(base_ok),
        "answer_ok": bool(answer_ok),
        "placement_ok": bool(placement_ok),
        "cons_only": bool(cons_only),
        "cons_ok": bool(base_ok and cons_only),
        "strict_cons_ok": bool(base_ok and placement_ok and cons_only),
        "per_cons": per,
        "per_cons_all": per_all,
        "da_extra": da_extra,
        "da_trace": da_trace,
        "da_other": da_other,
        "cot_trace_ok": bool(cot_trace_ok),
        "copy_flag": bool(copy_flag),
        "n_lines": len(lines),
        "n_commas": t.count(","),
    }


def load_results(path=OUT_RAW, stage=None):
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    if stage:
        rows = [r for r in rows if r.get("stage") == stage]
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    sc = [score(r) for r in rows]
    rescored = ("base_ok", "answer_ok", "placement_ok", "cons_only", "cons_ok", "strict_cons_ok",
                "per_cons", "per_cons_all", "da_extra", "da_trace", "da_other",
                "cot_trace_ok", "copy_flag", "n_lines", "n_commas")
    df = df.drop(columns=[c for c in rescored if c in df.columns])
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(sc)], axis=1)

## 8. 正式单约束校准

`F0` + 12 条候选，各 50 题，DA/CoT 两种模式。共 1300 条。
可中断；重跑本格会续跑。

In [8]:
CONS_LISTS = [["F0"]] + [[c] for c in CANDIDATES]
jobs = make_jobs("cons_calib", QUESTIONS, MODES, CONS_LISTS)
print("jobs:", len(jobs), "| 12 x 50 x 2 + F0 =", len(jobs))
run_jobs(jobs)

cal = load_results(stage="cons_calib")
print("已完成行数:", len(cal))
print(cal.groupby(["cons_id", "mode"]).size().unstack("mode").to_string())

jobs: 1300 | 12 x 50 x 2 + F0 = 1300
总计 1300，已完成 0，待跑 1300
(EngineCore pid=205) WARNING 11-07 15:18:39 [jit_monitor.py:140] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=205) WARNING 11-07 15:18:39 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _topp_sb_stats_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=205) WARNING 11-07 15:18:40 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _topp_sb_step_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=205) WARNING 11-07 15:18:42 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _topp_sb_mask_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
[DA] 64/650  batch 30.9s
[DA] 128/650  batch 10.0s
[DA] 192/650 

## 9. 汇总、三门槛与淘汰理由

`cons_pass` 使用 `strict_cons_ok = base_ok and placement_ok and cons_only`，格式或顺序错乱直接判失败。
另外单列 `cons_value`（值对）、`base_ok`、`placement`、`da_trace`，用来区分「值错」「位置错」「DA 抄表」。
- `F0(DA) pass`：要求型约束是「主动生成率」；禁令型约束取 `1 - pass` 作为「自然违反率」。
- `gap = pass_DA - pass_CoT`；`gap_slope` 检查成本是否随 d 上升。
- `structure_ok` 防止把格式崩坏误判成约束能力；`da_other_max` 不含 `X -> Y` 步骤行。

**注意**：`UPPER` 在 DA 下答案标签通常已经大写，F0 主动生成率可能接近 1，按门槛会被淘汰；这是预期行为。
`LOWER` 和 `UPPER` 互斥，选择时最多留一条。

In [9]:
BASE_MIN = 0.90
DA_PASS_MIN = 0.95
GAP_MIN = 0.15
GAP_SLOPE_MAX = 0.02
TRUNC_MAX = 0.05
COPY_MAX = 0.01
DA_EXTRA_MAX = 0.20


def _pass_all(d, c):
    return bool(d.get(c, False)) if isinstance(d, dict) else False


def summarize(cal):
    if cal.empty:
        raise RuntimeError("没有校准结果。先运行第 8 格。")
    cal = cal.copy()
    cal["track"] = cal["cons_id"].map(lambda c: CONS.get(c, {}).get("track", "?"))

    g = cal.groupby(["cons_id", "mode"])
    summ = g.agg(
        n=("id", "size"),
        cons_pass=("strict_cons_ok", "mean"),
        cons_value=("cons_ok", "mean"),
        cons_only=("cons_only", "mean"),
        base_ok=("base_ok", "mean"),
        placement=("placement_ok", "mean"),
        answer=("answer_ok", "mean"),
        trunc=("trunc", "mean"),
        da_extra=("da_extra", "mean"),
        da_trace=("da_trace", "mean"),
        da_other=("da_other", "mean"),
        copy=("copy_flag", "mean"),
        n_tokens=("n_tokens", "mean"),
        tok_p95=("n_tokens", lambda s: s.quantile(0.95)),
        n_lines=("n_lines", "median"),
    )
    summ["cons_pass_stop"] = cal[~cal["trunc"].astype(bool)].groupby(["cons_id", "mode"])["strict_cons_ok"].mean()
    summ = summ.round(3)

    pv = summ["cons_pass_stop"].unstack("mode").reindex(columns=["DA", "CoT"])
    bo = summ["base_ok"].unstack("mode").reindex(columns=["DA", "CoT"])
    pl = summ["placement"].unstack("mode").reindex(columns=["DA", "CoT"])
    tr = summ["trunc"].unstack("mode").reindex(columns=["DA", "CoT"])
    cp = summ["copy"].unstack("mode").reindex(columns=["DA", "CoT"])
    de = summ["da_other"].unstack("mode").reindex(columns=["DA", "CoT"])
    dt = summ["da_trace"].unstack("mode").reindex(columns=["DA", "CoT"])

    f0_da = cal[(cal["cons_id"] == "F0") & (cal["mode"] == "DA")]
    f0_pass = {}
    for c in ALL_CANDIDATES:
        f0_pass[c] = float(f0_da["per_cons_all"].map(lambda d: _pass_all(d, c)).mean()) if len(f0_da) else float("nan")

    rows = []
    candidates_run = [c for c in ALL_CANDIDATES if c in pv.index]
    for c in candidates_run:
        kind = CONS[c]["kind"]
        fp = f0_pass.get(c, float("nan"))
        non_idle_value = (1.0 - fp) if kind == "ban" else fp
        non_idle = (non_idle_value >= 0.15) if kind == "ban" else (non_idle_value <= 0.90)

        pass_da = float(pv.loc[c, "DA"]) if pd.notna(pv.loc[c, "DA"]) else float("nan")
        pass_cot = float(pv.loc[c, "CoT"]) if pd.notna(pv.loc[c, "CoT"]) else float("nan")
        gap = pass_da - pass_cot

        sub = cal[cal["cons_id"] == c]
        gd = sub.pivot_table(index="d", columns="mode", values="strict_cons_ok")
        if {"DA", "CoT"}.issubset(set(gd.columns)) and len(gd) >= 2:
            gap_by_d = (gd["DA"] - gd["CoT"]).astype(float)
            gap_slope = float(np.polyfit(gd.index.astype(float), gap_by_d.values, 1)[0])
        else:
            gap_slope = float("nan")
        gap_stable = bool(pd.notna(gap_slope) and gap_slope <= GAP_SLOPE_MAX)

        base_min = float(bo.loc[c].min(skipna=True))
        placement_min = float(pl.loc[c].min(skipna=True))
        trunc_max = float(tr.loc[c].max(skipna=True))
        copy_max = float(cp.loc[c].max(skipna=True))
        da_other_max = float(de.loc[c].max(skipna=True))
        da_trace_max = float(dt.loc[c].max(skipna=True))
        structure_ok = (base_min >= BASE_MIN and placement_min >= BASE_MIN and trunc_max < TRUNC_MAX and
                        copy_max <= COPY_MAX and da_other_max <= DA_EXTRA_MAX)

        failed = []
        if not structure_ok:
            failed.append("结构(base=%.3f,place=%.3f,trunc=%.3f,copy=%.3f,da_other=%.3f)" % (
                base_min, placement_min, trunc_max, copy_max, da_other_max))
        if not non_idle:
            failed.append("非空转(F0=%.3f)" % non_idle_value)
        if not (pd.notna(pass_da) and pass_da >= DA_PASS_MIN):
            failed.append("DA通过率=%.3f" % pass_da)
        if not (pd.notna(gap) and gap >= GAP_MIN):
            failed.append("CoT成本gap=%.3f" % gap)
        if not gap_stable:
            failed.append("gap随d上升(slope=%.3f)" % gap_slope)

        rows.append({
            "cons_id": c,
            "track": CONS[c]["track"],
            "kind": kind,
            "f0_da_pass": fp,
            "non_idle_value": non_idle_value,
            "non_idle": non_idle,
            "pass_DA": pass_da,
            "pass_CoT": pass_cot,
            "gap": gap,
            "gap_slope": gap_slope,
            "gap_stable": gap_stable,
            "base_min": base_min,
            "placement_min": placement_min,
            "trunc_max": trunc_max,
            "copy_max": copy_max,
            "da_other_max": da_other_max,
            "da_trace_max": da_trace_max,
            "structure_ok": structure_ok,
            "gates_ok": bool(structure_ok and non_idle and pd.notna(pass_da) and
                             pass_da >= DA_PASS_MIN and pd.notna(gap) and gap >= GAP_MIN and gap_stable),
            "reason": "; ".join(failed) if failed else "通过",
        })

    verdict = pd.DataFrame(rows).set_index("cons_id")
    print("逐约束汇总（cons_pass_stop 已排除截断）:")
    display(summ)
    print("\n门槛判定:")
    display(verdict.round(3))
    return summ, verdict

## 10. 逻辑独立性 + 选 10 条

两条互补检查：

1. **F0 文本**：任务 B.3 的原方法。禁令类有效；要求型在 F0 里几乎不出现，`n_A < 10` 时只能标 `NaN`。
2. **单约束输出交叉检查**：在 A 的单约束输出里看 B 是否顺便通过。若 `P(B 不通过 | A 通过) < 0.05`，加 B 基本免费，判冗余。

选择规则：先过全部门槛，按 `gap` 从大到小；检查双向蕴含；`LOWER`/`UPPER` 互斥；最后取前 10 条。
`k` 轴顺序：格式轨优先，查表轨其次，样板行最后；同轨内按 gap 从大到小。

In [10]:
def conditional_matrix(df, candidates):
    p = pd.DataFrame(index=candidates, columns=candidates, dtype=float)
    n = pd.DataFrame(index=candidates, columns=candidates, dtype=int)
    if df is None or len(df) == 0:
        return p, n
    for a in candidates:
        pass_a = df["per_cons_all"].map(lambda d: _pass_all(d, a))
        n_a = int(pass_a.sum())
        for b in candidates:
            if a == b:
                continue
            n.loc[a, b] = n_a
            if n_a >= 10:
                viol_b = ~df["per_cons_all"].map(lambda d: _pass_all(d, b))
                p.loc[a, b] = float((pass_a & viol_b).sum() / n_a)
    return p, n


def conditional_cross(cal, candidates):
    p = pd.DataFrame(index=candidates, columns=candidates, dtype=float)
    n = pd.DataFrame(index=candidates, columns=candidates, dtype=int)
    for a in candidates:
        sub = cal[cal["cons_id"] == a]
        if sub.empty:
            continue
        pass_a = sub["per_cons_all"].map(lambda d: _pass_all(d, a))
        n_a = int(pass_a.sum())
        for b in candidates:
            if a == b:
                continue
            n.loc[a, b] = n_a
            if n_a >= 10:
                viol_b = ~sub["per_cons_all"].map(lambda d: _pass_all(d, b))
                p.loc[a, b] = float((pass_a & viol_b).sum() / n_a)
    return p, n


def _json_default(o):
    if hasattr(o, "item"):
        return o.item()
    return str(o)


def select_constraints(verdict, indep_f0, indep_f0_n, indep_cross, indep_cross_n):
    passed = verdict[verdict["gates_ok"]].copy()
    passed = passed.sort_values(["gap", "pass_DA"], ascending=False)
    selected, dropped = [], []
    matrices = (("F0", indep_f0, indep_f0_n), ("cross", indep_cross, indep_cross_n))
    for c in passed.index:
        redundant_with = None
        for a in selected:
            if frozenset((a, c)) in INCOMPATIBLE:
                redundant_with = "与 %s 互斥" % a
                break
            for source, pmat, nmat in matrices:
                if not (a in pmat.index and c in pmat.columns):
                    continue
                p_ab = pmat.loc[a, c]   # P(B fail | A pass)
                n_ab = nmat.loc[a, c]
                p_ba = pmat.loc[c, a]   # P(A fail | B pass)
                n_ba = nmat.loc[c, a]
                if pd.notna(p_ab) and pd.notna(n_ab) and n_ab >= 10 and p_ab < 0.05:
                    redundant_with = ("%s: %s 蕴含 %s (p=%.3f, n=%s)" % (source, a, c, p_ab, n_ab))
                    break
                if pd.notna(p_ba) and pd.notna(n_ba) and n_ba >= 10 and p_ba < 0.05:
                    redundant_with = ("%s: %s 蕴含 %s (p=%.3f, n=%s)" % (source, c, a, p_ba, n_ba))
                    break
            if redundant_with:
                break
        if redundant_with:
            dropped.append((c, "冗余/互斥: " + redundant_with))
        else:
            selected.append(c)
    if len(selected) > TARGET_K:
        for c in selected[TARGET_K:]:
            dropped.append((c, "超过 k=%s 上限" % TARGET_K))
        selected = selected[:TARGET_K]
    return selected, dropped


def finalize(cal):
    summ, verdict = summarize(cal)
    all_run = [c for c in ALL_CANDIDATES if c in verdict.index]
    f0 = cal[cal["cons_id"] == "F0"]
    indep_f0, indep_f0_n = conditional_matrix(f0, all_run)
    indep_cross, indep_cross_n = conditional_cross(cal, all_run)
    selected, dropped = select_constraints(verdict, indep_f0, indep_f0_n, indep_cross, indep_cross_n)

    track_order = {"format": 0, "lookup": 1, "boilerplate": 2}
    k_order = sorted(selected, key=lambda c: (track_order.get(CONS[c]["track"], 9), -float(verdict.loc[c, "gap"])))

    summ.to_csv(OUT_SUMMARY, encoding="utf-8")
    indep_f0.to_csv(OUT_INDEP_F0, encoding="utf-8")
    indep_cross.to_csv(OUT_INDEP_CROSS, encoding="utf-8")

    drop_reason = dict(dropped)
    with open(OUT_VERDICTS, "w", encoding="utf-8") as f:
        for c in ALL_CANDIDATES:
            if c in verdict.index:
                rec = verdict.loc[c].to_dict()
                rec["cons_id"] = c
                if c in selected:
                    rec["status"] = "selected"
                elif c in drop_reason:
                    rec["status"] = "rejected"
                    rec["reason"] = drop_reason[c]
                elif c in BACKUP_CANDIDATES:
                    rec["status"] = "backup_unused"
                else:
                    rec["status"] = "rejected"
            else:
                rec = {"cons_id": c, "status": "not_run", "reason": "backup 未运行"}
            f.write(json.dumps(rec, ensure_ascii=False, default=_json_default) + "\n")

    payload = {
        "prompt_version": PROMPT_VERSION,
        "model": HF_ID,
        "dataset": DATASET_OUT,
        "n_questions": len(QUESTIONS),
        "question_qids": [q["qid"] for q in QUESTIONS],
        "selected": selected,
        "k_order": k_order,
        "dropped": [{"cons_id": c, "reason": r} for c, r in dropped],
        "constraints": {
            c: {
                "track": CONS[c]["track"],
                "kind": CONS[c]["kind"],
                "rule": CONS[c]["text"],
                "stats": verdict.loc[c].to_dict(),
            }
            for c in selected
        },
    }
    with open(OUT_SELECTED, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=_json_default)

    print("\n入选（%s/%s）:" % (len(selected), TARGET_K), selected)
    print("k 轴顺序:", k_order)
    print("被剔除:")
    for c, r in dropped:
        print("  -", c, ":", r)
    print("\n输出文件:")
    for p in (OUT_RAW, OUT_SUMMARY, OUT_VERDICTS, OUT_INDEP_F0, OUT_INDEP_CROSS, OUT_SELECTED):
        print("  ", p)
    return summ, verdict, indep_f0, indep_cross, selected, k_order, dropped


summ, verdict, indep_f0, indep_cross, SELECTED, K_ORDER, DROPPED = finalize(cal)

逐约束汇总（cons_pass_stop 已排除截断）:


n  cons_pass  cons_value  cons_only  base_ok  placement  \
cons_id     mode                                                             
D           CoT   50       0.98        0.98       0.98     1.00       1.00   
            DA    50       0.98        0.98       0.98     1.00       1.00   
F0          CoT   50       1.00        1.00       1.00     1.00       1.00   
            DA    50       1.00        1.00       1.00     1.00       1.00   
FANIN_START CoT   50       0.12        0.12       0.12     1.00       1.00   
            DA    50       0.56        0.56       0.56     1.00       1.00   
LASTROW     CoT   50       0.04        0.04       0.04     1.00       0.90   
            DA    50       0.04        0.06       0.06     0.98       0.70   
LOWER       CoT   50       0.02        0.02       0.02     1.00       1.00   
            DA    50       0.00        0.00       0.00     0.98       0.98   
NEXT        CoT   50       0.24        0.40       0.40     1.00       0.84   
            DA    50       0.36        0.36       0.40     0.92       0.92   
NO_COMMA    CoT   50       1.00        1.00       1.00     1.00       1.00   
            DA    50       1.00        1.00       1.00     1.00       1.00   
NO_EQUALS   CoT   50       0.98        0.98       0.98     1.00       1.00   
            DA    50       0.88        0.88       0.88     1.00       1.00   
PAREN       CoT   50       0.00        0.00       0.00     1.00       1.00   
            DA    50       0.00        0.00       0.00     1.00       1.00   
START       CoT   50       1.00        1.00       1.00     1.00       1.00   
            DA    50       1.00        1.00       1.00     1.00       1.00   
STATUS      CoT   50       0.60        1.00       1.00     1.00       0.60   
            DA    50       0.90        0.98       0.98     1.00       0.92   
TABLE_SIZE  CoT   50       0.00        0.00       0.00     0.86       0.22   
            DA    50       0.18        0.18       0.18     1.00       1.00   
UPPER       CoT   50       1.00        1.00       1.00     1.00       1.00   
            DA    50       1.00        1.00       1.00     1.00       1.00   

                  answer  trunc  da_extra  da_trace  da_other  copy  n_tokens  \
cons_id     mode                                                                
D           CoT     1.00   0.00       0.0      0.00      0.00   0.0     57.28   
            DA      0.06   0.00       1.0      0.02      0.98   0.0    100.02   
F0          CoT     1.00   0.00       0.0      0.00      0.00   0.0     82.90   
            DA      0.82   0.00       1.0      0.98      0.12   0.0     78.24   
FANIN_START CoT     0.98   0.00       0.0      0.00      0.00   0.0     53.78   
            DA      0.14   0.00       1.0      0.06      1.00   0.0     34.92   
LASTROW     CoT     0.96   0.00       0.0      0.00      0.00   0.0     59.34   
            DA      0.08   0.02       1.0      0.08      1.00   0.0     44.16   
LOWER       CoT     1.00   0.00       0.0      0.00      0.00   0.0     67.80   
            DA      0.10   0.02       1.0      0.02      1.00   0.0     21.00   
NEXT        CoT     0.88   0.00       0.0      0.00      0.00   0.0     67.46   
            DA      0.06   0.04       1.0      0.16      1.00   0.0     70.32   
NO_COMMA    CoT     1.00   0.00       0.0      0.00      0.00   0.0     62.78   
            DA      0.08   0.00       1.0      0.00      1.00   0.0     17.08   
NO_EQUALS   CoT     1.00   0.00       0.0      0.00      0.00   0.0     64.46   
            DA      0.12   0.00       1.0      0.12      1.00   0.0     20.54   
PAREN       CoT     1.00   0.00       0.0      0.00      0.00   0.0     68.70   
            DA      0.02   0.00       1.0      0.02      1.00   0.0     18.84   
START       CoT     0.92   0.00       0.0      0.00      0.00   0.0     42.32   
            DA      0.58   0.00       1.0      0.46      0.54   0.0     36.52   
STATUS      CoT     0.64   0.00       0.0      0.00 


门槛判定:


,track,kind,f0_da_pass,non_idle_value,non_idle,pass_DA,pass_CoT,gap,gap_slope,gap_stable,base_min,placement_min,trunc_max,copy_max,da_other_max,da_trace_max,structure_ok,gates_ok,reason
cons_id,,,,,,,,,,,,,,,,,,,
LOWER,format,require,0.00,0.00,True,0.000,0.02,-0.020,-0.004,True,0.98,0.98,0.02,0.0,1.00,0.02,False,False,"结构(base=0.980,place=0.980,trunc=0.020,copy=0.0..."
UPPER,format,require,0.98,0.98,False,1.000,1.00,0.000,0.000,True,1.00,1.00,0.00,0.0,1.00,0.02,False,False,"结构(base=1.000,place=1.000,trunc=0.000,copy=0.0..."
PAREN,format,require,0.00,0.00,True,0.000,0.00,0.000,0.000,True,1.00,1.00,0.00,0.0,1.00,0.02,False,False,"结构(base=1.000,place=1.000,trunc=0.000,copy=0.0..."
NO_COMMA,format,ban,0.98,0.02,False,1.000,1.00,0.000,0.000,True,1.00,1.00,0.00,0.0,1.00,0.00,False,False,"结构(base=1.000,place=1.000,trunc=0.000,copy=0.0..."
NO_EQUALS,format,ban,0.98,0.02,False,0.880,0.98,-0.100,-0.034,True,1.00,1.00,0.00,0.0,1.00,0.12,False,False,"结构(base=1.000,place=1.000,trunc=0.000,copy=0.0..."
START,lookup,require,0.00,0.00,True,1.000,1.00,0.000,0.000,True,1.00,1.00,0.00,0.0,0.54,0.46,False,False,"结构(base=1.000,place=1.000,trunc=0.000,copy=0.0..."
D,lookup,require,0.00,0.00,True,0.980,0.98,0.000,0.000,True,1.00,1.00,0.00,0.0,0.98,0.02,False,False,"结构(base=1.000,place=1.000,trunc=0.000,copy=0.0..."
NEXT,lookup,require,0.00,0.00,True,0.375,0.24,0.135,-0.147,True,0.92,0.84,0.04,0.0,1.00,0.16,False,False,"结构(base=0.920,place=0.840,trunc=0.040,copy=0.0..."
LASTROW,lookup,require,0.00,0.00,True,0.041,0.04,0.001,0.027,False,0.98,0.70,0.02,0.0,1.00,0.08,False,False,"结构(base=0.980,place=0.700,trunc=0.020,copy=0.0..."



入选（0/10）: []
k 轴顺序: []
被剔除:

输出文件:
   /kaggle/working/calib_v4_cons.jsonl
   /kaggle/working/calib_v4_cons_summary.csv
   /kaggle/working/calib_v4_cons_verdicts.jsonl
   /kaggle/working/calib_v4_cons_independence_f0.csv
   /kaggle/working/calib_v4_cons_independence_cross.csv
   /kaggle/working/calib_v4_cons_selected.json


## 11. 不足 10 条时自动补跑 backup

`SECOND / ODD_D / FIRSTROW / FANIN_NEXT` 是备份池。只有正式 12 条不足 10 条时才跑。
`run_jobs` 会跳过已完成的行；重复运行本格安全。

In [11]:
AUTO_BACKUP = True

if AUTO_BACKUP and len(SELECTED) < TARGET_K:
    print("正式候选只入选 %s 条，补跑 %s 条 backup。" % (len(SELECTED), len(BACKUP_CANDIDATES)))
    run_jobs(make_jobs("cons_calib", QUESTIONS, MODES, [[b] for b in BACKUP_CANDIDATES]))
    cal = load_results(stage="cons_calib")
    summ, verdict, indep_f0, indep_cross, SELECTED, K_ORDER, DROPPED = finalize(cal)
else:
    print("不需要补跑 backup（当前入选 %s 条）。" % len(SELECTED))

正式候选只入选 0 条，补跑 4 条 backup。
总计 400，已完成 0，待跑 400
[DA] 64/200  batch 19.4s
[DA] 128/200  batch 14.5s
[DA] 192/200  batch 13.1s
[DA] 200/200  batch 1.6s
[CoT] 32/200  batch 13.0s
[CoT] 64/200  batch 65.9s
[CoT] 96/200  batch 10.5s
[CoT] 128/200  batch 10.1s
[CoT] 160/200  batch 35.3s
[CoT] 192/200  batch 14.2s
[CoT] 200/200  batch 10.0s
逐约束汇总（cons_pass_stop 已排除截断）:


n  cons_pass  cons_value  cons_only  base_ok  placement  \
cons_id     mode                                                             
D           CoT   50       0.98        0.98       0.98     1.00       1.00   
            DA    50       0.98        0.98       0.98     1.00       1.00   
F0          CoT   50       1.00        1.00       1.00     1.00       1.00   
            DA    50       1.00        1.00       1.00     1.00       1.00   
FANIN_NEXT  CoT   50       0.32        0.40       0.40     0.98       0.64   
            DA    50       0.52        0.52       0.52     1.00       1.00   
FANIN_START CoT   50       0.12        0.12       0.12     1.00       1.00   
            DA    50       0.56        0.56       0.56     1.00       1.00   
FIRSTROW    CoT   50       0.00        0.00       0.00     1.00       1.00   
            DA    50       0.00        0.00       0.00     1.00       1.00   
LASTROW     CoT   50       0.04        0.04       0.04     1.00       0.90   
            DA    50       0.04        0.06       0.06     0.98       0.70   
LOWER       CoT   50       0.02        0.02       0.02     1.00       1.00   
            DA    50       0.00        0.00       0.00     0.98       0.98   
NEXT        CoT   50       0.24        0.40       0.40     1.00       0.84   
            DA    50       0.36        0.36       0.40     0.92       0.92   
NO_COMMA    CoT   50       1.00        1.00       1.00     1.00       1.00   
            DA    50       1.00        1.00       1.00     1.00       1.00   
NO_EQUALS   CoT   50       0.98        0.98       0.98     1.00       1.00   
            DA    50       0.88        0.88       0.88     1.00       1.00   
ODD_D       CoT   50       0.58        0.92       0.92     1.00       0.60   
            DA    50       0.56        0.56       0.58     0.98       0.98   
PAREN       CoT   50       0.00        0.00       0.00     1.00       1.00   
            DA    50       0.00        0.00       0.00     1.00       1.00   
SECOND      CoT   50       0.02        0.02       0.02     0.98       1.00   
            DA    50       0.02        0.02       0.06     0.74       0.74   
START       CoT   50       1.00        1.00       1.00     1.00       1.00   
            DA    50       1.00        1.00       1.00     1.00       1.00   
STATUS      CoT   50       0.60        1.00       1.00     1.00       0.60   
            DA    50       0.90        0.98       0.98     1.00       0.92   
TABLE_SIZE  CoT   50       0.00        0.00       0.00     0.86       0.22   
            DA    50       0.18        0.18       0.18     1.00       1.00   
UPPER       CoT   50       1.00        1.00       1.00     1.00       1.00   
            DA    50       1.00        1.00       1.00     1.00       1.00   

                  answer  trunc  da_extra  da_trace  da_other  copy  n_tokens  \
cons_id     mode                                                                
D           CoT     1.00   0.00       0.0      0.00      0.00  0.00     57.28   
            DA      0.06   0.00       1.0      0.02      0.98  0.00    100.02   
F0          CoT     1.00   0.00       0.0      0.00      0.00  0.00     82.90   
            DA      0.82   0.00       1.0      0.98      0.12  0.00     78.24   
FANIN_NEXT  CoT     0.82   0.00       0.0      0.00      0.00  0.00     81.24   
            DA      0.02   0.00       1.0      0.00      1.00  0.00     18.36   
FANIN_START CoT     0.98   0.00       0.0      0.00      0.00  0.00     53.78   
            DA      0.14   0.00       1.0      0.06      1.00  0.00     34.92   
FIRSTROW    CoT     0.98   0.00       0.0      0.00      0.00  0.00     44.48   
            DA      0.38   0.00       1.0      0.46      0.68  0.00     46.18   
LASTROW     CoT     0.96   0.00       0.0      0.00      0.00  0.00     59.34   
            DA      0.08   0.02       1.0      0.08      1.00  0.00     44.16   
LOWER       CoT     1.00   0.00       0.0      0.00      0.00  0.00     67.8


门槛判定:


,track,kind,f0_da_pass,non_idle_value,non_idle,pass_DA,pass_CoT,gap,gap_slope,gap_stable,base_min,placement_min,trunc_max,copy_max,da_other_max,da_trace_max,structure_ok,gates_ok,reason
cons_id,,,,,,,,,,,,,,,,,,,
LOWER,format,require,0.00,0.00,True,0.000,0.02,-0.020,-0.004,True,0.98,0.98,0.02,0.00,1.00,0.02,False,False,"结构(base=0.980,place=0.980,trunc=0.020,copy=0.0..."
UPPER,format,require,0.98,0.98,False,1.000,1.00,0.000,0.000,True,1.00,1.00,0.00,0.00,1.00,0.02,False,False,"结构(base=1.000,place=1.000,trunc=0.000,copy=0.0..."
PAREN,format,require,0.00,0.00,True,0.000,0.00,0.000,0.000,True,1.00,1.00,0.00,0.00,1.00,0.02,False,False,"结构(base=1.000,place=1.000,trunc=0.000,copy=0.0..."
NO_COMMA,format,ban,0.98,0.02,False,1.000,1.00,0.000,0.000,True,1.00,1.00,0.00,0.00,1.00,0.00,False,False,"结构(base=1.000,place=1.000,trunc=0.000,copy=0.0..."
NO_EQUALS,format,ban,0.98,0.02,False,0.880,0.98,-0.100,-0.034,True,1.00,1.00,0.00,0.00,1.00,0.12,False,False,"结构(base=1.000,place=1.000,trunc=0.000,copy=0.0..."
START,lookup,require,0.00,0.00,True,1.000,1.00,0.000,0.000,True,1.00,1.00,0.00,0.00,0.54,0.46,False,False,"结构(base=1.000,place=1.000,trunc=0.000,copy=0.0..."
D,lookup,require,0.00,0.00,True,0.980,0.98,0.000,0.000,True,1.00,1.00,0.00,0.00,0.98,0.02,False,False,"结构(base=1.000,place=1.000,trunc=0.000,copy=0.0..."
NEXT,lookup,require,0.00,0.00,True,0.375,0.24,0.135,-0.147,True,0.92,0.84,0.04,0.00,1.00,0.16,False,False,"结构(base=0.920,place=0.840,trunc=0.040,copy=0.0..."
LASTROW,lookup,require,0.00,0.00,True,0.041,0.04,0.001,0.027,False,0.98,0.70,0.02,0.00,1.00,0.08,False,False,"结构(base=0.980,place=0.700,trunc=0.020,copy=0.0..."



入选（0/10）: []
k 轴顺序: []
被剔除:

输出文件:
   /kaggle/working/calib_v4_cons.jsonl
   /kaggle/working/calib_v4_cons_summary.csv
   /kaggle/working/calib_v4_cons_verdicts.jsonl
   /kaggle/working/calib_v4_cons_independence_f0.csv
   /kaggle/working/calib_v4_cons_independence_cross.csv
   /kaggle/working/calib_v4_cons_selected.json


## 12. 人工审查清单（必须做）

自动判分只能筛候选。请逐项看下面的样本：

- [ ] 每条入选约束在 DA 下是否只输出要求的标签行 + 最终行，没有偷写推理。
- [ ] 每条入选约束在 CoT 下是否真的多写一行/一段，成本来自约束本身，而不是截断。
- [ ] 查表标签是否只出现一次，数值是否和题面一致（`NEXT` / `SECOND` / `LASTROW` / `FANIN_*` 最容易错）。
- [ ] `base_ok` 是否接近 1；如果某条约束的 `base_ok` 低，先修 prompt，不要看通过率。
- [ ] 截断率是否集中在高 d；如果是，提高 `max_tokens` 或缩短 trace 要求，不要硬截断。
- [ ] F0 自然违反率是否符合预期：`NO_COMMA / NO_EQUALS / UPPER` 应该被非空转门槛淘汰。
- [ ] `LOWER` 与 `UPPER` 不能同时入选；如果同时出现，检查选择逻辑。
- [ ] 10 条约束是否两两独立；如果两条总同时通过/失败，考虑换掉一条。

> 这一步不能省。任务 B 的结论要进 `record.md` 和 `AGENTS.md`，错误入选会污染合并复核。

In [12]:
# 逐条打印 2 个样本，DA/CoT 都看
for c in ["F0"] + CANDIDATES + BACKUP_CANDIDATES:
    for mode in MODES:
        sub = cal[(cal["cons_id"] == c) & (cal["mode"] == mode)]
        if sub.empty:
            continue
        print("=" * 90)
        print("%s | %s | n=%s | strict=%.3f | value=%.3f | base=%.3f | place=%.3f | trunc=%.3f | da_trace=%.3f | da_other=%.3f" % (
            c, mode, len(sub), sub["strict_cons_ok"].mean(), sub["cons_ok"].mean(),
            sub["base_ok"].mean(), sub["placement_ok"].mean(),
            sub["trunc"].mean(), sub["da_trace"].mean(), sub["da_other"].mean()))
        for _, r in sub.sample(min(2, len(sub)), random_state=SEED).iterrows():
            print("-" * 60)
            print(r["text"][:1200])

print("\n按 d 的 cons_pass:")
display(cal.pivot_table(index=["cons_id", "mode"], columns="d", values="strict_cons_ok").round(2))

print("\nF0 基线（答案正确率 / token / 行数），只作诊断，不作为任务 B 指标:")
display(cal[cal["cons_id"] == "F0"].pivot_table(
    index="mode", columns="d", values=["answer_ok", "n_tokens", "n_lines"]).round(2))

print("\nF0(DA) 自然违反率 / 主动生成率:")
f0_da = cal[(cal["cons_id"] == "F0") & (cal["mode"] == "DA")]
f0_rows = []
for c in ALL_CANDIDATES:
    fp = f0_da["per_cons_all"].map(lambda d: _pass_all(d, c)).mean()
    f0_rows.append({
        "cons_id": c,
        "kind": CONS[c]["kind"],
        "F0_DA_pass": fp,
        "non_idle_value": (1 - fp) if CONS[c]["kind"] == "ban" else fp,
    })
display(pd.DataFrame(f0_rows).set_index("cons_id").round(3))

F0 | DA | n=50 | strict=1.000 | value=1.000 | base=1.000 | place=1.000 | trunc=0.000 | da_trace=0.980 | da_other=0.120
------------------------------------------------------------
X -> T  
T -> W  
W -> L  
L -> D  
D -> F  
####ANSWER=F
------------------------------------------------------------
V -> S  
S -> F  
F -> R  
R -> G  
G -> K  
K -> M  
M -> A  
A -> X  
X -> J  
####ANSWER=J
F0 | CoT | n=50 | strict=1.000 | value=1.000 | base=1.000 | place=1.000 | trunc=0.000 | da_trace=0.000 | da_other=0.000
------------------------------------------------------------
Start at X.

Step 1: X -> T  
Entry reached: T

Step 2: T -> W  
Entry reached: W

Step 3: W -> L  
Entry reached: L

Step 4: L -> D  
Entry reached: D

Step 5: D -> F  
Entry reached: F

####ANSWER=F
------------------------------------------------------------
Start at V.

Step 1: V -> S  
Step 2: S -> F  
Step 3: F -> R  
Step 4: R -> G  
Step 5: G -> K  
Step 6: K -> M  
Step 7: M -> A  
Step 8: A -> X  
Step 9: X -> J 

d                   5     6     7     8     9     10
cons_id     mode                                    
D           CoT   1.00  1.00  1.00  1.00  1.00  0.89
            DA    1.00  1.00  1.00  1.00  1.00  0.89
F0          CoT   1.00  1.00  1.00  1.00  1.00  1.00
            DA    1.00  1.00  1.00  1.00  1.00  1.00
FANIN_NEXT  CoT   0.25  0.25  0.38  0.38  0.22  0.44
            DA    0.50  0.25  0.62  0.62  0.56  0.56
FANIN_START CoT   0.00  0.25  0.25  0.25  0.00  0.00
            DA    0.38  0.62  0.50  0.75  0.67  0.44
FIRSTROW    CoT   0.00  0.00  0.00  0.00  0.00  0.00
            DA    0.00  0.00  0.00  0.00  0.00  0.00
LASTROW     CoT   0.12  0.00  0.00  0.12  0.00  0.00
            DA    0.00  0.00  0.12  0.00  0.00  0.11
LOWER       CoT   0.00  0.00  0.00  0.12  0.00  0.00
            DA    0.00  0.00  0.00  0.00  0.00  0.00
NEXT        CoT   0.00  0.25  0.00  0.50  0.22  0.44
            DA    0.62  0.50  0.25  0.38  0.11  0.33
NO_COMMA    CoT   1.00  1.00  1.00  1.00  1.00  1.00
            DA    1.00  1.00  1.00  1.00  1.00  1.00
NO_EQUALS   CoT   1.00  1.00  0.88  1.00  1.00  1.00
            DA    1.00  0.88  0.88  0.88  0.89  0.78
ODD_D       CoT   0.38  0.38  0.75  0.38  0.89  0.67
            DA    0.00  1.00  0.25  1.00  0.11  1.00
PAREN       CoT   0.00  0.00  0.00  0.00  0.00  0.00
            DA    0.00  0.00  0.00  0.00  0.00  0.00
SECOND      CoT   0.00  0.00  0.12  0.00  0.00  0.00
            DA    0.00  0.00  0.00  0.00  0.00  0.11
START       CoT   1.00  1.00  1.00  1.00  1.00  1.00
            DA    1.00  1.00  1.00  1.00  1.00  1.00
STATUS      CoT   0.50  0.25  0.62  0.75  0.67  0.78
            DA    1.00  0.88  0.88  1.00  0.89  0.78
TABLE_SIZE  CoT   0.00  0.00  0.00  0.00  0.00  0.00
            DA    0.25  0.38  0.00  0.00  0.11  0.33
UPPER       CoT   1.00  1.00  1.00  1.00  1.00  1.00
            DA    1.00  1.00  1.00  1.00  1.00  1.00


F0 基线（答案正确率 / token / 行数），只作诊断，不作为任务 B 指标:


answer_ok                             n_lines                             \
d           5     6     7    8    9     10      5     6      7      8      9    
mode                                                                            
CoT        1.0  1.00  1.00  1.0  1.0  1.00    12.0  12.5  13.75  12.50  12.00   
DA         1.0  0.88  0.88  0.5  1.0  0.67    18.0  22.0  13.12  26.38  12.67   

            n_tokens                                      
d        10       5      6      7       8      9      10  
mode                                                      
CoT   12.44    74.25  77.12  88.88   81.75  86.22  88.11  
DA    23.33    73.25  88.00  53.62  112.00  52.11  92.00


F0(DA) 自然违反率 / 主动生成率:


,kind,F0_DA_pass,non_idle_value
cons_id,,,
LOWER,require,0.00,0.00
UPPER,require,0.98,0.98
PAREN,require,0.00,0.00
NO_COMMA,ban,0.98,0.02
NO_EQUALS,ban,0.98,0.02
START,require,0.00,0.00
D,require,0.00,0.00
NEXT,require,0.00,0.00
LASTROW,require,0.00,0.00


## 13. 下一步

1. 把 `calib_v4_cons_selected.json` 里的约束固定下来，写进 `record.md` 的「v3 约束集」。
2. 合并复核：新任务 × 10 约束 × 2 模式 × 100 题 = 2000 条，确认三门槛仍成立。
3. `k = 1..10` 累积曲线，`d ∈ {1,5,10,15,20}`，每格 50 题。
4. 分析 `gap` 随 d 与 k 的变化、`leak_rate`、截断率、`da_trace` 比例。
5. 回填 `record.md` 与 `AGENTS.md`。

**再次提醒：先人工看第 12 节的样本，再相信自动筛选结果。**